# Vision-Language Models Fundamentals

**Module 12: Multimodal AI**  
**Objective**: Understanding vision-language models from scratch

Multimodal AI:
- Process multiple modalities (text, images, audio, video)
- Vision-language models (VLMs)
- Contrastive learning (CLIP)
- Image-text alignment
- Zero-shot classification

## What You'll Learn
1. Multimodal architectures
2. CLIP (Contrastive Language-Image Pre-training)
3. Vision-language alignment
4. Zero-shot image classification
5. Image-text retrieval
6. Building VLMs from scratch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. What is Multimodal AI?

**Multimodal** = Multiple data types (modalities)

Common modalities:
- **Vision**: Images, video
- **Language**: Text
- **Audio**: Speech, music
- **Other**: Sensor data, graphs, etc.

**Goal**: Unified representation across modalities

**Why multimodal?**
- Humans perceive world through multiple senses
- Richer understanding from combined signals
- Cross-modal tasks (image captioning, VQA)

In [ ]:
def visualize_multimodal_landscape():
    """Visualize multimodal AI landscape."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Modality types
    ax1 = axes[0, 0]
    ax1.axis('off')
    ax1.set_title('Modalities', fontsize=14, weight='bold', pad=20)
    
    modalities = [
        ("Vision", "Images/Video", "CNNs, ViT", 0.8, 0.7, 'lightblue'),
        ("Language", "Text", "Transformers", 0.2, 0.7, 'lightgreen'),
        ("Audio", "Speech/Music", "Wav2Vec", 0.8, 0.3, 'lightyellow'),
        ("Other", "Sensors/Graphs", "Custom", 0.2, 0.3, 'lightcoral')
    ]
    
    for name, data, model, x, y, color in modalities:
        circle = plt.Circle((x, y), 0.12, color=color, ec='black', linewidth=2)
        ax1.add_patch(circle)
        ax1.text(x, y+0.05, name, ha='center', va='center', fontsize=11, weight='bold')
        ax1.text(x, y-0.02, data, ha='center', va='center', fontsize=8)
        ax1.text(x, y-0.08, f"({model})", ha='center', va='center', fontsize=7, style='italic')
    
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    # 2. Fusion strategies
    ax2 = axes[0, 1]
    ax2.axis('off')
    ax2.set_title('Fusion Strategies', fontsize=14, weight='bold', pad=20)
    
    fusion_y = [0.8, 0.5, 0.2]
    fusion_types = [
        ("Early Fusion", "Combine raw inputs", "Simple but rigid"),
        ("Late Fusion", "Combine predictions", "Independent processing"),
        ("Hybrid Fusion", "Multi-stage combination", "Best of both")
    ]
    
    for i, (name, desc, note) in enumerate(fusion_types):
        y = fusion_y[i]
        rect = plt.Rectangle((0.1, y-0.08), 0.8, 0.15, 
                            color=['lightblue', 'lightgreen', 'lightyellow'][i],
                            ec='black', linewidth=2)
        ax2.add_patch(rect)
        ax2.text(0.5, y+0.02, name, ha='center', va='center', fontsize=11, weight='bold')
        ax2.text(0.5, y-0.03, desc, ha='center', va='center', fontsize=8)
        ax2.text(0.5, y-0.06, note, ha='center', va='center', fontsize=7, style='italic')
    
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    # 3. Vision-Language Tasks
    ax3 = axes[1, 0]
    ax3.axis('off')
    ax3.set_title('Vision-Language Tasks', fontsize=14, weight='bold', pad=20)
    
    tasks = [
        "Image Captioning",
        "VQA (Visual Q&A)",
        "Image-Text Retrieval",
        "Zero-Shot Classification",
        "Visual Grounding",
        "Image Generation"
    ]
    
    for i, task in enumerate(tasks):
        y = 0.9 - i * 0.15
        ax3.text(0.1, y, f"{i+1}. {task}", fontsize=10, va='center')
        rect = plt.Rectangle((0.7, y-0.03), 0.2, 0.06, 
                            color='lightblue', ec='black', linewidth=1)
        ax3.add_patch(rect)
    
    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    
    # 4. Key Models Timeline
    ax4 = axes[1, 1]
    ax4.set_title('Evolution of VLMs', fontsize=14, weight='bold', pad=10)
    
    timeline = [
        (2021, "CLIP", 8),
        (2022, "DALL-E 2", 6),
        (2022, "Flamingo", 4),
        (2023, "GPT-4V", 9),
        (2023, "LLaVA", 5),
        (2024, "Gemini", 10)
    ]
    
    years = [t[0] for t in timeline]
    models = [t[1] for t in timeline]
    scores = [t[2] for t in timeline]
    
    colors_map = plt.cm.viridis(np.linspace(0.3, 0.9, len(timeline)))
    bars = ax4.bar(range(len(models)), scores, color=colors_map, edgecolor='black', linewidth=1.5)
    
    ax4.set_xticks(range(len(models)))
    ax4.set_xticklabels([f"{m}\n({y})" for y, m, _ in timeline], fontsize=9)
    ax4.set_ylabel('Capability Score', fontsize=10)
    ax4.set_ylim(0, 12)
    ax4.grid(axis='y', alpha=0.3, linestyle='--')
    
    for i, (bar, score) in enumerate(zip(bars, scores)):
        ax4.text(bar.get_x() + bar.get_width()/2, score + 0.3, 
                str(score), ha='center', va='bottom', fontsize=9, weight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nMultimodal AI Overview:")
    print("="*70)
    print("\nKey Concepts:")
    print("  1. Alignment: Map different modalities to shared space")
    print("  2. Contrastive Learning: Pull similar pairs together, push apart dissimilar")
    print("  3. Zero-shot: Classify without task-specific training")
    print("  4. Embedding: Dense vector representations")
    print("\nApplications:")
    print("  • Search: Find images from text queries")
    print("  • Captioning: Generate descriptions of images")
    print("  • VQA: Answer questions about images")
    print("  • Generation: Create images from text (DALL-E, Stable Diffusion)")

visualize_multimodal_landscape()

## 2. CLIP Architecture

**CLIP** (Contrastive Language-Image Pre-training) by OpenAI (2021)

**Key Innovation**: Learn vision-language alignment from 400M image-text pairs

**Architecture**:
```
Image Encoder (Vision Transformer or ResNet)
    ↓
Image Embedding (512-dim)
    
Text Encoder (Transformer)
    ↓
Text Embedding (512-dim)

Contrastive Loss: Align matching pairs
```

**Training**:
- Batch of N (image, text) pairs
- Create N×N similarity matrix
- Maximize diagonal (correct pairs)
- Minimize off-diagonal (incorrect pairs)

In [ ]:
class ImageEncoder(nn.Module):
    """
    Simple image encoder using CNN.
    
    In production, use Vision Transformer (ViT) or ResNet.
    """
    
    def __init__(self, embed_dim: int = 512):
        super().__init__()
        
        # Simple CNN for demonstration
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.projection = nn.Linear(512, embed_dim)
        
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: (batch_size, 3, H, W)
            
        Returns:
            embeddings: (batch_size, embed_dim)
        """
        features = self.conv_layers(images)  # (B, 512, 1, 1)
        features = features.view(features.size(0), -1)  # (B, 512)
        embeddings = self.projection(features)  # (B, embed_dim)
        
        # L2 normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return embeddings

class TextEncoder(nn.Module):
    """
    Simple text encoder using transformer.
    
    In production, use full BERT or GPT-style model.
    """
    
    def __init__(self, vocab_size: int = 10000, embed_dim: int = 512, 
                 max_len: int = 77, n_layers: int = 6, n_heads: int = 8):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Parameter(torch.randn(max_len, embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=2048,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.projection = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, text_tokens: torch.Tensor) -> torch.Tensor:
        """
        Args:
            text_tokens: (batch_size, seq_len)
            
        Returns:
            embeddings: (batch_size, embed_dim)
        """
        seq_len = text_tokens.size(1)
        embeddings = self.token_embedding(text_tokens)  # (B, seq_len, embed_dim)
        embeddings = embeddings + self.position_embedding[:seq_len]
        
        encoded = self.transformer(embeddings)  # (B, seq_len, embed_dim)
        
        # Take [CLS] token (first token) or mean pooling
        pooled = encoded[:, 0, :]  # (B, embed_dim)
        
        projected = self.projection(pooled)  # (B, embed_dim)
        
        # L2 normalize
        projected = F.normalize(projected, p=2, dim=1)
        
        return projected

class CLIPModel(nn.Module):
    """
    CLIP: Contrastive Language-Image Pre-training.
    
    Learns joint embedding space for images and text.
    """
    
    def __init__(self, embed_dim: int = 512, vocab_size: int = 10000):
        super().__init__()
        
        self.image_encoder = ImageEncoder(embed_dim)
        self.text_encoder = TextEncoder(vocab_size, embed_dim)
        
        # Learnable temperature parameter
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        
    def forward(self, images: torch.Tensor, text_tokens: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            images: (batch_size, 3, H, W)
            text_tokens: (batch_size, seq_len)
            
        Returns:
            image_embeds: (batch_size, embed_dim)
            text_embeds: (batch_size, embed_dim)
        """
        image_embeds = self.image_encoder(images)
        text_embeds = self.text_encoder(text_tokens)
        
        return image_embeds, text_embeds
    
    def compute_loss(self, images: torch.Tensor, text_tokens: torch.Tensor) -> torch.Tensor:
        """
        Compute contrastive loss.
        
        Args:
            images: (batch_size, 3, H, W)
            text_tokens: (batch_size, seq_len)
            
        Returns:
            loss: scalar
        """
        image_embeds, text_embeds = self.forward(images, text_tokens)
        
        # Compute similarity matrix
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_embeds @ text_embeds.t()
        logits_per_text = logits_per_image.t()
        
        # Targets: diagonal is correct pairing
        batch_size = images.size(0)
        targets = torch.arange(batch_size, device=images.device)
        
        # Cross-entropy loss (symmetric)
        loss_image = F.cross_entropy(logits_per_image, targets)
        loss_text = F.cross_entropy(logits_per_text, targets)
        
        loss = (loss_image + loss_text) / 2
        
        return loss

# Create CLIP model
clip_model = CLIPModel(embed_dim=512, vocab_size=10000).to(device)

total_params = sum(p.numel() for p in clip_model.parameters())
image_params = sum(p.numel() for p in clip_model.image_encoder.parameters())
text_params = sum(p.numel() for p in clip_model.text_encoder.parameters())

print("CLIP Model Architecture:")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"  Image encoder: {image_params:,}")
print(f"  Text encoder: {text_params:,}")

# Test forward pass
batch_size = 4
images = torch.randn(batch_size, 3, 224, 224).to(device)
text_tokens = torch.randint(0, 10000, (batch_size, 77)).to(device)

with torch.no_grad():
    image_embeds, text_embeds = clip_model(images, text_tokens)
    
print(f"\nForward pass:")
print(f"  Input images: {images.shape}")
print(f"  Input text: {text_tokens.shape}")
print(f"  Image embeddings: {image_embeds.shape}")
print(f"  Text embeddings: {text_embeds.shape}")

# Compute similarity
similarities = (image_embeds @ text_embeds.t()).cpu()
print(f"\nSimilarity matrix: {similarities.shape}")
print("Diagonal (correct pairs) should have highest scores after training")
print(similarities)

## 3. Contrastive Learning

**Contrastive Loss** pulls similar pairs together, pushes dissimilar apart.

**InfoNCE Loss** (used in CLIP):

$$
\mathcal{L}_{\text{NCE}} = -\log \frac{\exp(\text{sim}(i, t_i) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(i, t_j) / \tau)}
$$

Where:
- $i$ = image embedding
- $t_i$ = corresponding text embedding
- $\tau$ = temperature (controls sharpness)
- $N$ = batch size

In [ ]:
def visualize_contrastive_learning():
    """Visualize contrastive learning process."""
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # 1. Before training
    ax1 = axes[0]
    ax1.set_title('Before Training\n(Random embeddings)', fontsize=12, weight='bold')
    ax1.set_xlim(-1.5, 1.5)
    ax1.set_ylim(-1.5, 1.5)
    ax1.set_xlabel('Dimension 1')
    ax1.set_ylabel('Dimension 2')
    ax1.grid(True, alpha=0.3)
    
    # Random scattered points
    np.random.seed(42)
    images_x = np.random.randn(5) * 0.8
    images_y = np.random.randn(5) * 0.8
    texts_x = np.random.randn(5) * 0.8
    texts_y = np.random.randn(5) * 0.8
    
    ax1.scatter(images_x, images_y, s=200, c='blue', marker='s', 
               label='Images', edgecolors='black', linewidths=2, alpha=0.7)
    ax1.scatter(texts_x, texts_y, s=200, c='red', marker='o', 
               label='Texts', edgecolors='black', linewidths=2, alpha=0.7)
    
    for i in range(5):
        ax1.text(images_x[i], images_y[i], f'I{i+1}', ha='center', va='center', fontsize=9, weight='bold')
        ax1.text(texts_x[i], texts_y[i], f'T{i+1}', ha='center', va='center', fontsize=9, weight='bold')
    
    ax1.legend(loc='upper right')
    
    # 2. During training
    ax2 = axes[1]
    ax2.set_title('During Training\n(Learning alignment)', fontsize=12, weight='bold')
    ax2.set_xlim(-1.5, 1.5)
    ax2.set_ylim(-1.5, 1.5)
    ax2.set_xlabel('Dimension 1')
    ax2.set_ylabel('Dimension 2')
    ax2.grid(True, alpha=0.3)
    
    # Partially aligned
    images_x = np.array([0.8, -0.6, 0.3, -0.7, 0.5])
    images_y = np.array([0.6, 0.7, -0.8, -0.5, 0.9])
    texts_x = images_x + np.random.randn(5) * 0.2
    texts_y = images_y + np.random.randn(5) * 0.2
    
    # Draw arrows showing pull
    for i in range(5):
        ax2.arrow(texts_x[i], texts_y[i], 
                 (images_x[i] - texts_x[i]) * 0.6, 
                 (images_y[i] - texts_y[i]) * 0.6,
                 head_width=0.08, head_length=0.06, fc='green', ec='green', alpha=0.5, lw=2)
    
    ax2.scatter(images_x, images_y, s=200, c='blue', marker='s', 
               label='Images', edgecolors='black', linewidths=2, alpha=0.7)
    ax2.scatter(texts_x, texts_y, s=200, c='red', marker='o', 
               label='Texts', edgecolors='black', linewidths=2, alpha=0.7)
    
    for i in range(5):
        ax2.text(images_x[i], images_y[i], f'I{i+1}', ha='center', va='center', fontsize=9, weight='bold')
        ax2.text(texts_x[i], texts_y[i], f'T{i+1}', ha='center', va='center', fontsize=9, weight='bold')
    
    ax2.legend(loc='upper right')
    
    # 3. After training
    ax3 = axes[2]
    ax3.set_title('After Training\n(Aligned embeddings)', fontsize=12, weight='bold')
    ax3.set_xlim(-1.5, 1.5)
    ax3.set_ylim(-1.5, 1.5)
    ax3.set_xlabel('Dimension 1')
    ax3.set_ylabel('Dimension 2')
    ax3.grid(True, alpha=0.3)
    
    # Fully aligned
    images_x = np.array([0.8, -0.6, 0.3, -0.7, 0.5])
    images_y = np.array([0.6, 0.7, -0.8, -0.5, 0.9])
    texts_x = images_x + np.random.randn(5) * 0.05  # Very close
    texts_y = images_y + np.random.randn(5) * 0.05
    
    ax3.scatter(images_x, images_y, s=200, c='blue', marker='s', 
               label='Images', edgecolors='black', linewidths=2, alpha=0.7)
    ax3.scatter(texts_x, texts_y, s=200, c='red', marker='o', 
               label='Texts', edgecolors='black', linewidths=2, alpha=0.7)
    
    # Draw circles showing pairs
    for i in range(5):
        circle = plt.Circle((images_x[i], images_y[i]), 0.15, color='green', 
                          fill=False, linestyle='--', linewidth=2, alpha=0.6)
        ax3.add_patch(circle)
        ax3.text(images_x[i], images_y[i], f'I{i+1}', ha='center', va='center', fontsize=9, weight='bold')
        ax3.text(texts_x[i], texts_y[i], f'T{i+1}', ha='center', va='center', fontsize=9, weight='bold')
    
    ax3.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    print("\nContrastive Learning Process:")
    print("="*70)
    print("\n1. BEFORE: Embeddings are random, no alignment")
    print("   • Images and texts scattered independently")
    print("   • No semantic relationship captured")
    
    print("\n2. DURING: Contrastive loss pulls matching pairs together")
    print("   • Positive pairs (I1,T1) attract")
    print("   • Negative pairs (I1,T2) repel")
    print("   • Temperature τ controls sharpness")
    
    print("\n3. AFTER: Matched pairs occupy same region in embedding space")
    print("   • 'cat' photo near 'cat' text")
    print("   • 'dog' photo near 'dog' text")
    print("   • Enable zero-shot classification!")

visualize_contrastive_learning()

# Demonstrate similarity matrix evolution
def simulate_similarity_matrices():
    """Show similarity matrix before and after training."""
    
    N = 5
    
    # Before: random
    torch.manual_seed(42)
    sim_before = torch.randn(N, N) * 0.3
    
    # After: high diagonal
    sim_after = torch.randn(N, N) * 0.2
    sim_after = sim_after - torch.diag(torch.diag(sim_after))
    sim_after = sim_after + torch.eye(N) * 2.0
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    im1 = ax1.imshow(sim_before.numpy(), cmap='RdYlGn', vmin=-1, vmax=2)
    ax1.set_title('Similarity Matrix: Before Training', fontsize=12, weight='bold')
    ax1.set_xlabel('Text Index')
    ax1.set_ylabel('Image Index')
    ax1.set_xticks(range(N))
    ax1.set_yticks(range(N))
    ax1.set_xticklabels([f'T{i+1}' for i in range(N)])
    ax1.set_yticklabels([f'I{i+1}' for i in range(N)])
    
    for i in range(N):
        for j in range(N):
            ax1.text(j, i, f'{sim_before[i,j]:.2f}', 
                    ha='center', va='center', fontsize=9)
    
    plt.colorbar(im1, ax=ax1, label='Similarity')
    
    im2 = ax2.imshow(sim_after.numpy(), cmap='RdYlGn', vmin=-1, vmax=2)
    ax2.set_title('Similarity Matrix: After Training', fontsize=12, weight='bold')
    ax2.set_xlabel('Text Index')
    ax2.set_ylabel('Image Index')
    ax2.set_xticks(range(N))
    ax2.set_yticks(range(N))
    ax2.set_xticklabels([f'T{i+1}' for i in range(N)])
    ax2.set_yticklabels([f'I{i+1}' for i in range(N)])
    
    for i in range(N):
        for j in range(N):
            ax2.text(j, i, f'{sim_after[i,j]:.2f}', 
                    ha='center', va='center', fontsize=9,
                    weight='bold' if i == j else 'normal')
    
    plt.colorbar(im2, ax=ax2, label='Similarity')
    
    plt.tight_layout()
    plt.show()
    
    # Compute losses
    targets = torch.arange(N)
    tau = 0.07
    logits_before = sim_before / tau
    logits_after = sim_after / tau
    
    loss_before = F.cross_entropy(logits_before, targets)
    loss_after = F.cross_entropy(logits_after, targets)
    
    print("\nSimilarity Matrix Analysis:")
    print("="*70)
    print("\nBEFORE: Random similarities, no pattern")
    print(f"  Contrastive Loss: {loss_before.item():.4f}")
    
    print("\nAFTER: High diagonal (correct pairs), low off-diagonal")
    print(f"  Contrastive Loss: {loss_after.item():.4f}")
    print(f"  Improvement: {((loss_before - loss_after) / loss_before * 100).item():.1f}%")

simulate_similarity_matrices()

## 4. Zero-Shot Classification

CLIP enables **zero-shot** image classification without training.

**How it works**:
1. Encode image
2. Encode candidate class names as text ("a photo of a dog", "a photo of a cat")
3. Compute similarities
4. Pick class with highest similarity

In [ ]:
class ZeroShotClassifier:
    """
    Zero-shot image classifier using CLIP.
    """
    
    def __init__(self, clip_model: CLIPModel):
        self.clip_model = clip_model
        self.clip_model.eval()
        
    def classify(self, image: torch.Tensor, class_names: List[str], 
                 text_template: str = "a photo of a {}") -> Tuple[str, torch.Tensor]:
        """
        Classify image into one of the given classes.
        
        Args:
            image: (3, H, W)
            class_names: List of class names
            text_template: Template for text encoding
            
        Returns:
            predicted_class: str
            probabilities: (num_classes,)
        """
        with torch.no_grad():
            # Encode image
            image_batch = image.unsqueeze(0).to(device)  # (1, 3, H, W)
            image_embed = self.clip_model.image_encoder(image_batch)  # (1, embed_dim)
            
            # Encode class names
            text_prompts = [text_template.format(name) for name in class_names]
            
            # Simulate tokenization (in production, use proper tokenizer)
            text_tokens = torch.randint(0, 10000, (len(class_names), 77)).to(device)
            text_embeds = self.clip_model.text_encoder(text_tokens)  # (num_classes, embed_dim)
            
            # Compute similarities
            similarities = (image_embed @ text_embeds.t()).squeeze(0)  # (num_classes,)
            
            # Convert to probabilities
            probabilities = F.softmax(similarities * 100, dim=0)  # Temperature scaling
            
            # Get prediction
            pred_idx = probabilities.argmax().item()
            predicted_class = class_names[pred_idx]
            
            return predicted_class, probabilities.cpu()
    
    def retrieve_images(self, text_query: str, image_embeddings: torch.Tensor, 
                       top_k: int = 5) -> torch.Tensor:
        """
        Retrieve top-k images matching text query.
        
        Args:
            text_query: Text description
            image_embeddings: (N, embed_dim) precomputed image embeddings
            top_k: Number of images to retrieve
            
        Returns:
            indices: (top_k,) indices of top matches
        """
        with torch.no_grad():
            # Encode text
            text_tokens = torch.randint(0, 10000, (1, 77)).to(device)
            text_embed = self.clip_model.text_encoder(text_tokens)  # (1, embed_dim)
            
            # Compute similarities
            similarities = (text_embed @ image_embeddings.t()).squeeze(0)  # (N,)
            
            # Get top-k
            top_indices = similarities.topk(top_k).indices
            
            return top_indices.cpu()

# Test zero-shot classification
classifier = ZeroShotClassifier(clip_model)

# Simulate image
test_image = torch.randn(3, 224, 224)

# Class names
classes = ["dog", "cat", "bird", "car", "airplane"]

# Classify
predicted, probs = classifier.classify(test_image, classes)

print("Zero-Shot Image Classification:")
print("="*70)
print(f"\nPredicted class: {predicted}")
print("\nClass probabilities:")
for class_name, prob in zip(classes, probs):
    bar = "█" * int(prob * 50)
    print(f"  {class_name:10s} {prob:.4f} {bar}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(classes, probs.numpy(), color='skyblue', edgecolor='black', linewidth=1.5)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Zero-Shot Classification Probabilities', fontsize=14, weight='bold')
ax.set_ylim(0, max(probs.numpy()) * 1.2)

for bar, prob in zip(bars, probs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{prob:.3f}', ha='center', va='bottom', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Zero-shot classification works without any task-specific training!")
print("  Model learned general image-text alignment from web data")

## 5. Image-Text Retrieval

CLIP enables bidirectional search:
- **Text → Image**: Find images matching text query
- **Image → Text**: Find captions matching image

Both use the same embedding space!

In [ ]:
def demonstrate_retrieval():
    """Demonstrate image-text retrieval."""
    
    # Simulate database of images and texts
    num_items = 10
    
    # Generate random embeddings (in production, these would be real CLIP embeddings)
    torch.manual_seed(42)
    image_embeds = F.normalize(torch.randn(num_items, 512), p=2, dim=1).to(device)
    text_embeds = F.normalize(torch.randn(num_items, 512), p=2, dim=1).to(device)
    
    image_labels = [f"Image_{i}" for i in range(num_items)]
    text_labels = [f"Text_{i}" for i in range(num_items)]
    
    # Text-to-Image retrieval
    print("TEXT-TO-IMAGE RETRIEVAL")
    print("="*70)
    
    query_text_idx = 2
    query_embed = text_embeds[query_text_idx:query_text_idx+1]  # (1, 512)
    
    # Compute similarities
    similarities = (query_embed @ image_embeds.t()).squeeze(0)  # (num_items,)
    
    # Get top-3
    top_k = 3
    top_indices = similarities.topk(top_k).indices.cpu()
    top_scores = similarities.topk(top_k).values.cpu()
    
    print(f"\nQuery: {text_labels[query_text_idx]}")
    print(f"\nTop {top_k} matching images:")
    for rank, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        print(f"  {rank}. {image_labels[idx]} (similarity: {score:.4f})")
    
    # Image-to-Text retrieval
    print("\n\nIMAGE-TO-TEXT RETRIEVAL")
    print("="*70)
    
    query_image_idx = 5
    query_embed = image_embeds[query_image_idx:query_image_idx+1]  # (1, 512)
    
    # Compute similarities
    similarities = (query_embed @ text_embeds.t()).squeeze(0)  # (num_items,)
    
    # Get top-3
    top_indices = similarities.topk(top_k).indices.cpu()
    top_scores = similarities.topk(top_k).values.cpu()
    
    print(f"\nQuery: {image_labels[query_image_idx]}")
    print(f"\nTop {top_k} matching texts:")
    for rank, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        print(f"  {rank}. {text_labels[idx]} (similarity: {score:.4f})")
    
    # Visualize similarity matrix
    similarity_matrix = (image_embeds @ text_embeds.t()).cpu()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(similarity_matrix.numpy(), cmap='viridis', aspect='auto')
    ax.set_xlabel('Text Index', fontsize=12)
    ax.set_ylabel('Image Index', fontsize=12)
    ax.set_title('Image-Text Similarity Matrix', fontsize=14, weight='bold')
    
    ax.set_xticks(range(num_items))
    ax.set_yticks(range(num_items))
    ax.set_xticklabels([f'T{i}' for i in range(num_items)])
    ax.set_yticklabels([f'I{i}' for i in range(num_items)])
    
    plt.colorbar(im, ax=ax, label='Similarity')
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Same embedding space enables bidirectional retrieval!")

demonstrate_retrieval()

## 6. Training CLIP

Training loop for CLIP model.

In [ ]:
def train_clip(model, dataloader, num_epochs=3, lr=1e-4):
    """
    Train CLIP model with contrastive loss.
    
    Args:
        model: CLIPModel
        dataloader: DataLoader with (images, text_tokens) pairs
        num_epochs: Number of training epochs
        lr: Learning rate
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Cosine LR scheduler
    total_steps = num_epochs * len(dataloader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_steps)
    
    model.train()
    
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        
        for batch_idx, (images, text_tokens) in enumerate(dataloader):
            images = images.to(device)
            text_tokens = text_tokens.to(device)
            
            # Forward pass
            loss = model.compute_loss(images, text_tokens)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
            losses.append(loss.item())
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(dataloader)}, "
                      f"Loss: {loss.item():.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
        
        avg_loss = epoch_loss / len(dataloader)
        print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}\n")
    
    return losses

# Simulate training
print("CLIP Training Demo")
print("="*70)

# Create fake dataset
class FakeDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples=100):
        self.num_samples = num_samples
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        image = torch.randn(3, 224, 224)
        text_tokens = torch.randint(0, 10000, (77,))
        return image, text_tokens

dataset = FakeDataset(num_samples=50)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

# Train for 2 epochs
print("\nTraining CLIP model...")
print("(Using simulated data for demonstration)\n")

losses = train_clip(clip_model, dataloader, num_epochs=2, lr=1e-4)

# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(losses, linewidth=2)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Contrastive Loss', fontsize=12)
plt.title('CLIP Training Loss', fontsize=14, weight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Training complete!")
print(f"  Final loss: {losses[-1]:.4f}")
print(f"  Total steps: {len(losses)}")

## Summary

You've mastered vision-language models fundamentals:
- ✅ Multimodal AI concepts (fusion strategies, cross-modal tasks)
- ✅ CLIP architecture (image encoder, text encoder, contrastive loss)
- ✅ Contrastive learning (InfoNCE loss, similarity matrices)
- ✅ Building VLMs from scratch (encoders, projection heads)
- ✅ Zero-shot classification (no task-specific training needed)
- ✅ Image-text retrieval (bidirectional search in shared space)
- ✅ Training CLIP models (contrastive loss optimization)

**Key Insights**:
1. **Multimodal learning** aligns different modalities in shared embedding space
2. **CLIP** learns from 400M image-text pairs without manual labels
3. **Contrastive loss** pulls matching pairs together, pushes apart mismatches
4. **Zero-shot** classification works by computing text-image similarities
5. **Temperature** parameter τ controls sharpness of similarity distribution
6. **L2 normalization** of embeddings makes cosine similarity equivalent to dot product

**CLIP Advantages**:
- No manual labels needed (learns from web data)
- Zero-shot transfer to new tasks
- Robust to distribution shift
- Interpretable via natural language
- Unified representation for vision and language

**Applications**:
- Image search engines
- Content moderation
- Visual question answering
- Image captioning
- Cross-modal retrieval
- Zero-shot classification

**Next Notebook**: Advanced multimodal applications (LLaVA, image captioning, VQA, multimodal RAG)

## Exercises

1. **Implement Vision Transformer**: Replace CNN image encoder with ViT
2. **Hard negative mining**: Improve training with hard negative samples
3. **Multi-language CLIP**: Extend to support multiple languages
4. **Fine-tune CLIP**: Adapt pretrained CLIP for specific domain